# EDA — Presuntos Suicidios Colombia 2015-2024

Este notebook revisa la **calidad de los datos crudos (bronze)** y deja la evidencia de cada regla de limpieza que aplica el ETL. No modifica nada: todo se lee como texto (`dtype=str`), solo se detecta, se cuantifica y se interpreta.

El análisis descriptivo e inferencial (gráficos, intervalo de confianza y pruebas de hipótesis) está en el documento del trabajo y se hace en R a partir de la capa gold.

**Fuentes**
- `suicidios`: *Presuntos Suicidios. Colombia, 2015 a 2024. Cifras definitivas* (INMLCF), API de Datos Abiertos `f75u-mirk`.
- `poblacion_2005_2017` y `poblacion_2018_2050`: proyecciones de población departamental por área (DANE).

Antes de ejecutarlo: correr `python main.py` una vez (descarga la base a `data/bronze/`) y descargar los dos Excel del DANE en `data/bronze/`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent))   # para importar src/ desde notebooks/
from src.utils import load_config
from src.extract.extract_data import extract_source
from src.transform.clean_data import quitar_tildes

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)

config = load_config()


def clave(texto):
    """Forma normalizada para detectar la misma categoria escrita distinto:
    sin tildes, en minusculas y sin espacios ni signos."""
    if pd.isna(texto):
        return texto
    return "".join(c for c in quitar_tildes(str(texto)).lower() if c.isalnum())


def es_sin_info(serie):
    return serie.map(clave).eq("sininformacion")

# Parte A — Base de presuntos suicidios (INMLCF)

## 1. Carga y tamaño
Cada fila es un caso individual de presunto suicidio. Se esperan **26.558 registros** (la cifra publicada) y 32 columnas, con casos de 2015 a 2024.

In [ ]:
df = extract_source(config["sources"]["suicidios"])
print(df.shape)
df.head()

In [ ]:
df["a_o_del_hecho"].value_counts().sort_index()

## 2. Calidad: nulos y "Sin información"
En esta base el dato faltante **no viene vacío**: se registra como la categoría `Sin información` (a veces `Sin Información`). Por eso los nulos del sistema salen en cero y hay que contar esa categoría aparte.

In [ ]:
perfil = pd.DataFrame({
    "nulos": df.isna().sum(),
    "sin_info": df.apply(lambda c: es_sin_info(c).sum()),
    "unicos": df.nunique(),
})
perfil["pct_sin_info"] = (perfil["sin_info"] / len(df) * 100).round(1)
perfil.sort_values("sin_info", ascending=False)

Lo que se ve:
- No hay nulos de sistema en ninguna columna.
- `razon_del_suicidio` tiene 52,9 % de "Sin información", `escolaridad` 16,4 %, `estado_civil` 10,8 %, `escenario_del_hecho` 2,9 %, `zona_del_hecho` 0,6 % y `departamento_del_hecho_dane` 4 casos. Coincide con la Tabla 1 del documento.
- `rango_de_hora_del_hecho_x_3_horas` y `localidad_del_hecho` tienen la mayoría sin dato; no se usan en el análisis.
- `circunstancia_del_hecho_detallada` y `manera_de_muerte` tienen **un solo valor** en toda la base (son constantes): no aportan información.

**Regla:** "Sin información" se conserva como una categoría más (no se imputa ni se eliminan filas) y se unifica su escritura.

## 3. Duplicados

In [ ]:
print("ids repetidos:", df["id"].duplicated().sum())
print("filas idénticas (contando id):", df.duplicated().sum())
print("filas idénticas sin contar id:", df.drop(columns="id").duplicated().sum())

In [ ]:
# Ver el par de filas iguales en todo menos el id
iguales = df[df.drop(columns="id").duplicated(keep=False)]
iguales

El `id` es único. Solo hay **un par de registros** iguales en todas las columnas menos el `id`. Como cada fila es una persona y las variables son categorías amplias (grupo de edad, municipio, mes), dos personas distintas pueden coincidir en todo.

**Regla:** no se elimina ningún registro; la llave es `id`.

## 4. Categorías escritas distinto según el año
Para cada columna se agrupan los valores por su forma normalizada (sin tildes, minúsculas, sin espacios ni signos). Si un mismo grupo tiene más de una escritura, es la misma categoría registrada de formas distintas.

In [ ]:
COLUMNAS_CATEGORICAS = [c for c in df.columns if c != "id" and df[c].nunique() <= 100]

variantes = []
for col in COLUMNAS_CATEGORICAS:
    conteo = df[col].value_counts()
    grupos = conteo.groupby(conteo.index.map(clave))
    for _, g in grupos:
        if len(g) > 1:
            variantes.append({"columna": col, "variantes": " | ".join(g.index), "casos": g.sum()})

pd.DataFrame(variantes)

Para ver en qué años aparece cada escritura (ejemplo con algunas columnas):

In [ ]:
for col in ["mes_del_hecho", "dia_del_hecho", "departamento_del_hecho_dane", "zona_del_hecho",
            "grupo_mayor_menor_de_edad", "razon_del_suicidio", "mecanismo_causal_de_la_lesion_fatal"]:
    con_variantes = [v for fila in variantes if fila["columna"] == col for v in fila["variantes"].split(" | ")]
    if con_variantes:
        sub = df[df[col].isin(con_variantes)]
        display(pd.crosstab(sub[col], sub["a_o_del_hecho"]))

Lo que se ve:
- **Mes y día:** en 2021 y 2024 vienen con mayúscula inicial (`Enero`, `Lunes`); los demás años en minúscula.
- **Departamento:** `Bogotá D.C.` (2015-2016) y `Bogotá, D.C.`; `Quindio` (2015) y `Quindío`.
- **Zona:** `Centro poblado(corregimiento…)` y `Centro poblado (corregimiento…)` (2021-2024).
- **Mayor/menor de edad:** en 2015 cambia la redacción (`Mayor de Edad (>18 Años)`).
- **Razón del suicidio:** `Conflicto con pareja o ex-pareja` y `ex pareja` (2022-2023), `Escolares / educativas` con tres escrituras, `Abuso de sustancias / sustancia`, `Accesos a armas`, `Maltrato fisico…` sin tildes, entre otras.
- **Mecanismo causal:** `Cortopunzante` / `Corto punzante`, `Cortocontundente` / `Corto contundente`, tres formas de "explosivo".
- **Estado civil:** `Soltero(a)` / `Soltero (a)`, igual con casado y viudo.

**Regla:** unificar cada columna con `unificar_categorias` (se queda la escritura más frecuente) y mapear a mano los casos que cambian de redacción y no solo de tildes o espacios (explosivo, mayor/menor de edad).

In [ ]:
# Escolaridad: no son solo variantes de escritura, mezcla dos clasificaciones distintas
df["escolaridad"].value_counts()

`escolaridad` combina dos esquemas de clasificación según el año (por ejemplo `Básica primaria` y `Educación básica primaria`, `Preescolar` y `Educación inicial y educación preescolar`). No es una variable del análisis de asociación, así que solo se unifica la escritura y se deja anotado.

## 5. Mecanismo causal de la lesión fatal
El análisis de la paradoja de género recodifica el mecanismo en **violento / no violento** (anexo del documento).

In [ ]:
df["mecanismo_causal_de_la_lesion_fatal"].value_counts()

In [ ]:
VIOLENTO = ["generadoresdeasfixia", "proyectildearmadefuego", "contundente", "cortopunzante", "cortante",
            "cortocontundente", "punzante", "termico", "electrico", "agenteomecanismoexplosivo",
            "agentesymecanismoexplosivo", "mecanismooagenteexplosivo"]
NO_VIOLENTO = ["toxico", "caustico"]

mec = df["mecanismo_causal_de_la_lesion_fatal"].map(clave)
tipo = pd.Series("Por determinar", index=df.index)
tipo[mec.isin(VIOLENTO)] = "Violento"
tipo[mec.isin(NO_VIOLENTO)] = "No violento"

print("Sin clasificar:", sorted(set(mec[tipo.eq("Por determinar")])))
pd.crosstab(df["sexo_de_la_victima"], tipo, margins=True)

Todas las categorías quedan clasificadas, salvo `Por determinar` (35 casos), que se deja aparte. El resultado reproduce la Tabla 4 del documento: 22.288 violentos y 4.235 no violentos.

**Regla:** crear `tipo_mecanismo` (Violento / No violento / Por determinar) en gold.

## 6. Razón del suicidio
Variable con mayor subregistro. Se revisan las frecuencias entre los casos informados para decidir qué categorías se agrupan.

In [ ]:
razon = df["razon_del_suicidio"]
informados = razon[~es_sin_info(razon)]
print("informados:", len(informados), "de", len(razon))

frec = informados.map(clave).value_counts()
pd.DataFrame({"casos": frec, "pct_informados": (frec / len(informados) * 100).round(1)})

De los 12.500 casos con razón informada, nueve categorías concentran casi todo: conflicto de pareja, enfermedad física o mental, enfermedad mental, desamor, económicas, enfermedad física, abuso de sustancias, muerte de un familiar o amigo y jurídicas. Las demás tienen muy pocos casos (bullying, ciberacoso, orientación sexual diversa, acceso a armas, etc.) y producirían frecuencias esperadas menores a 5 en la prueba chi-cuadrado.

**Regla:** crear `razon_agrupada` con esas 9 categorías, `Otras` (las demás) y `Sin información`.

## 7. Departamentos y códigos DANE
La población del DANE se une por **código de departamento y año**, así que los códigos tienen que coincidir.

In [ ]:
pd.crosstab(df["codigo_dane_departamento"], df["departamento_del_hecho_dane"]).pipe(
    lambda t: t.loc[:, (t > 0).any()]
).apply(lambda fila: ", ".join(fila[fila > 0].index), axis=1).to_frame("nombres")

In [ ]:
df["codigo_dane_departamento"].str.len().value_counts()

Lo que se ve:
- 33 departamentos (incluye Bogotá) más `999` = "Sin información" (4 casos).
- Los códigos vienen **sin cero a la izquierda** (`5` Antioquia, `8` Atlántico), mientras que el DANE usa `05` y `08`.
- Un código puede tener dos nombres (Bogotá, Quindío), por eso conviene unir por código y no por nombre.

**Reglas:** rellenar el código a 2 dígitos (`zfill(2)`), tomar el nombre oficial del DANE y agregar la `region` (Amazonía, Andina, Caribe, Orinoquía, Pacífica) para el contraste por regiones. Los 4 casos sin departamento quedan fuera de las tasas departamentales.

# Parte B — Proyecciones de población DANE

Los archivos del DANE traen títulos arriba del encabezado, filas vacías y notas al final. El extractor busca la fila que empieza con `DP` (`header_starts` en `config.yaml`); en el archivo 2018-2050 los datos están en la hoja `PobDepartamentalxÁrea`.

In [ ]:
pob_a = extract_source(config["sources"]["poblacion_2005_2017"])
pob_b = extract_source(config["sources"]["poblacion_2018_2050"])
print(pob_a.shape, pob_a.columns.tolist())
print(pob_b.shape, pob_b.columns.tolist())
pob_b.head()

In [ ]:
for nombre, p in [("2005-2017", pob_a), ("2018-2050", pob_b)]:
    print(nombre)
    print("  años:", p["AÑO"].dropna().unique()[:3], "...", p["AÑO"].dropna().unique()[-3:])
    print("  áreas:", p["ÁREA GEOGRÁFICA"].dropna().unique())
    print("  códigos DP:", p["DP"].str.len().value_counts().to_dict())

# Notas al pie del archivo 2018-2050 (filas que no son de datos)
pob_b[~pob_b["DP"].str.fullmatch(r"\d{2}", na=False)]

Lo que se ve:
- Mismas columnas en ambos archivos excepto la de población: `Población` (2005-2017) y `TOTAL` (2018-2050).
- Cada departamento y año viene en tres filas: `Cabecera`, `Centros Poblados y Rural Disperso` y `Total`. Las cabeceras se llaman distinto en cada archivo (`Cabecera` / `Cabecera Municipal`).
- La población viene como texto con separador de miles (`4,037,095`).
- El archivo 2018-2050 trae al final notas de fuente que no son datos.
- Los códigos `DP` vienen con cero a la izquierda (`05`).

Comprobación: con la población nacional del DANE se obtienen las tasas del documento (4,47 en 2015 y 5,73 en 2024).

In [ ]:
def poblacion_total(p, col_pob):
    datos = p[p["ÁREA GEOGRÁFICA"].eq("Total") & p["DP"].str.fullmatch(r"\d{2}", na=False)].copy()
    datos["poblacion"] = pd.to_numeric(datos[col_pob].str.replace(",", "", regex=False))
    return datos.groupby("AÑO")["poblacion"].sum()

poblacion = pd.concat([poblacion_total(pob_a, "Población"), poblacion_total(pob_b, "TOTAL")])
casos = df["a_o_del_hecho"].value_counts()
tasas = pd.DataFrame({"casos": casos, "poblacion": poblacion}).dropna().astype({"casos": int, "poblacion": int})
tasas["tasa_100k"] = (tasas["casos"] / tasas["poblacion"] * 100_000).round(2)
tasas

# Conclusiones: de los hallazgos a las reglas de limpieza

| # | Hallazgo | Regla en el ETL | Capa |
|---|---|---|---|
| 1 | El dato faltante viene como "Sin información" / "Sin Información", no como nulo | Conservarlo como categoría y unificar su escritura; no imputar ni eliminar filas | Silver |
| 2 | Mes y día con mayúscula en 2021 y 2024; departamento, zona, estado civil, razón y mecanismo con variantes de tildes, espacios y guiones | `unificar_categorias` en cada columna categórica | Silver |
| 3 | Mayor/menor de edad (2015) y "explosivo" cambian de redacción | Mapeo manual a una sola categoría | Silver |
| 4 | `circunstancia_del_hecho_detallada` y `manera_de_muerte` son constantes | Se pueden excluir de gold | Silver |
| 5 | `id` único; un par de registros iguales en el resto de columnas | No eliminar registros | Silver |
| 6 | Códigos de departamento sin cero a la izquierda y nombres con variantes | `zfill(2)` y nombre oficial del DANE; agregar `region` | Silver / Gold |
| 7 | Mecanismo causal con 17 escrituras | `tipo_mecanismo`: Violento / No violento / Por determinar (anexo del documento) | Gold |
| 8 | Razón del suicidio con muchas categorías de pocos casos | `razon_agrupada`: 9 categorías + Otras + Sin información | Gold |
| 9 | DANE: títulos, filas vacías, notas al pie, columna de población con nombre distinto y separador de miles | Leer desde la fila `DP`, quedarse con área `Total`, convertir a número y unir 2015-2017 con 2018-2024 | Silver |
| 10 | La población permite calcular tasas por 100.000 hab. por año y por departamento | Tablas de tasas en gold (nacional y departamental) | Gold |